In [1]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 词汇表大小
    "context_length": 1024, # 上下文长度
    "emb_dim": 768, # 嵌入维度
    "n_heads": 12, # 注意力头数量
    "n_layers": 12, # 层数
    "drop_rate": 0.1,  # Dropout 率
    "qkv_bias": False # 查询-键-值偏置
}

In [2]:
# 一个包含占位符的 GPT 模型架构类
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # 词元嵌入
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        # 位置嵌入
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        # dropout
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # 使用占位符替换 Transformer 块
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        # 使用占位符替换最终层归一化
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        # 计算输入索引的词元和位置嵌入
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds

        # 应用 dropout
        x = self.drop_emb(x)

        # 通过 Transformer 块处理数据
        x = self.trf_blocks(x)

        # 应用归一化
        x = self.final_norm(x)
        
        # 使用线性输出层生成 logits
        logits = self.out_head(x)
        return logits

# Transformer 块占位符类
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # 一个简单占位符

    def forward(self, x):
        # 只返回其输入
        return x

# 最终层归一化占位符类
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # 这里的参数只是为了模仿归一化接口

    def forward(self, x):
        # 只返回其输入
        return x

In [3]:
# 由于模型内无分词器，所以需要写一个分词器后才能调用模型
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)


model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("\nOutput shape:", logits.shape)
print(logits)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.6340,  0.5614,  0.2352,  ..., -1.0327, -0.0042,  0.2117],
         [-1.4724, -0.1245,  0.6632,  ...,  0.6598,  0.6471, -0.9286],
         [-1.1892,  1.7514,  0.1999,  ...,  1.0686, -0.4218,  0.5248],
         [-0.3733, -0.8342,  0.6618,  ...,  1.4582, -0.3882,  0.4174]],

        [[-0.9126,  0.0502,  0.2022,  ..., -0.7278, -0.3322, -0.5955],
         [-1.0572, -0.0292,  0.2354,  ..., -0.0102, -0.1741, -1.0125],
         [-0.6738,  1.5192,  0.3688,  ...,  2.1352, -1.1685,  0.5792],
         [-0.0128, -0.8161,  0.4994,  ...,  0.7962, -0.5116,  0.3775]]],
       grad_fn=<UnsafeViewBackward0>)


虽然输出的形状是正确的，但由于模型目前使用的是随机初始化的权重，还没有经过大规模数据的预训练，所以这些输出数值目前是随机的 。简而言之，这个输出是模型在词汇空间里的“投票”，预训练的过程就是为了教会模型如何投出正确的那一票 。

层归一化的主要目的是通过**调整神经元激活值的分布**，使其保持均值为 0、方差为 1 的状态 。作用：
1. 稳定训练：这种调整有助于确保训练过程的一致性和可靠性 。  


2. 加速收敛：它能帮助权重更有效地收敛 。  


3. 独立性：与批归一化（Batch Norm）不同，它对每个输入独立进行归一化，不受批次大小限制，在处理大语言模型时更具灵活性 。

层归一化解决了“每一层输出太乱”的问题，从而让训练更稳定，但是它不能完全解决梯度消失等问题。

In [4]:
torch.manual_seed(123)

# 构建两个训练样本，每个样本包含 5 个维度（特征）
batch_example = torch.randn(2, 5) 

# 神经网络层包含一个线性层和一个非线性激活函数 ReLU (简单地将负输入值设为 0，从而确保层的输出值都是正值)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [5]:
# 具体实现作用在输入张量 x 的最后一个维度上，该维度对应于嵌入维度（emb_dim）
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        # 一个小常数 在归一化过程中会被加到方差上以防止除零错误
        self.eps = 1e-5

        # 两个可训练参数（与输入维度相同），调整它们以获得更好的模型性能
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        # 在进行均值或方差等运算时，使用 keepdim=True 可以确保输出张量与输入张量具有相同的维度。
        # dim 参数指定了在张量中计算统计量（如均值或方差）时应该沿着哪个维度进行。
        # unbiased=False 意味在方差计算中，使用样本数量 n 作为方差公式的除数（无贝塞尔修正，有偏方差）

        # 层归一化操作: 减去均值，并将结果除以方差的平方根（也就是标准差）
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

dim = 1 or -1 时计算列维度的平均值，以获得每行一个平均值；dim = 0 时计算行维度的平均值，以获得每列一个平均值。在 GPT 模型处理的三维张量 [batch_size, num_tokens, embedding_size] 中，使用 dim=-1 可以直接对最后一位（即嵌入特征维度）进行归一化，而不需要关心前面有多少个维度 。   

由于一个单词的特征通常存储在张量的最后一个维度（即每一行），我们需要使用 dim=-1 来确保归一化是针对“这一个单词的所有特征”进行的，而不是跨单词进行归一化 。

In [6]:
# 层归一化的应用示例
ln = LayerNorm(emb_dim=5) 
out_ln = ln(batch_example) 
mean = out_ln.mean(dim=-1, keepdim=True) 
var = out_ln.var(dim=-1, unbiased=False, keepdim=True) 
print("Mean:\n", mean) 
print("Variance:\n", var)

Mean:
 tensor([[-2.9802e-08],
        [ 0.0000e+00]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


层归一化与批归一化相比，层归一化是在特征维度上进行归一化，由于层归一化是对每个输入独立进行归一化，不受批次大小的限制，因此在这些场景中它提供了更多的灵活性和稳定性。这在分布式训练或在资源受限的环境中部署模型时尤为重要。

除了 ReLU 激活函数还有 GELU（Gaussian Error Linear Unit）和 SwiGLU（Swish-gated Linear Unit）等激活函数。GELU 和 SwiGLU 是更为复杂且平滑的激活函数，分别结合了高斯分布和 sigmoid 门控线性单元。与较为简单的 ReLU 激活函数相比，它们能够提升深度学习模型的性能。 GELU 激活函数可以通过多种方式实现，其精确的定义为 $$GELU(x) = x \cdot \Phi(x)$$ 其中 $\Phi(x)$ 是标准高斯分布的累积分布函数。计算量更小的近似公式为 $$GELU(x) \approx 0.5 \cdot x \cdot \left( 1 + \tanh \left[ \sqrt{\frac{2}{\pi}} \cdot (x + 0.044715 \cdot x^3) \right] \right)$$


In [7]:
# GELU激活函数实现
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

在神经网络（包括 GPT 模型）中，激活函数是不可或缺的。如果没有它，深度学习模型将失去处理复杂问题的能力。

以下是激活函数存在的意义及其具体作用：

### 1. 引入非线性（最核心的作用）
这是激活函数最根本的存在意义。
* **没有激活函数的情况**：神经网络中的每一层操作（矩阵乘法 $W \cdot x + b$）都是线性的。根据数学原理，**多个线性层的叠加等效于单个线性层**。无论你的模型堆叠了多少层，如果不加激活函数，它最终只能拟合简单的线性关系（如直线、平滑平面），无法处理复杂的现实数据。
* **有了激活函数的情况**：激活函数通过引入非线性（如 GELU、ReLU 的弯曲特性），允许模型将输入空间“扭曲”和“折叠”，从而能够拟合极其复杂的函数关系（如图像中的物体边缘、语言中的语义逻辑）。



### 2. 具体在模型中的作用

#### A. “门控”与特征选择
激活函数决定了哪些信息可以继续向下传递，哪些信息应该被抑制。
* 例如在 **ReLU** 中，负值会被直接设为 0，这相当于“关闭”了不相关的神经元，使网络具有**稀疏性**，提高了计算效率。
* 在 **GELU** 中，它以一种更平滑的方式决定权重的贡献度，这在 Transformer 架构中对捕捉细微语义非常有帮助。

#### B. 辅助模型训练（梯度传递）
神经网络是通过反向传播（梯度下降）来学习的。激活函数的导数（梯度）决定了权重更新的幅度。
* 合适的激活函数（如 GELU）在输入变化时能保持稳定的梯度流，防止模型在训练过程中陷入“死区”或者出现梯度消失。

#### C. 增强模型的表达能力
在教材 4.3 节提到的 Feed-Forward（前馈神经网路）中，模型先将维度扩大（例如从 768 维扩大到 3072 维），经过激活函数后再缩回。这个“先扩张-非线性变换-再收缩”的过程，本质上是让模型在更高维的空间里对信息进行非线性加工，提取出深层的特征。

### 总结
你可以把 **线性层** 想象成 **“搬运和组合零件”**，而把 **激活函数** 想象成 **“胶水和成型工具”**。没有激活函数，你只能把零件摆在一起；有了它，你才能把零件塑造成复杂的、具有智能行为的机器（GPT）。

In [8]:
# 利用 GELU 激活函数的前馈神经网络
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            # 两个线性层和一个 GELU 激活函数组成
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

print(GPT_CONFIG_124M["emb_dim"])

768


<div align="center">
    <img src="../pictures/GELUNN.png" width=70%>
</div>

In [9]:
ffn = FeedForward(GPT_CONFIG_124M)

# input shape: [batch_size, num_token, emb_size]
# 线性层 1 将输入投射到一个 4 倍大的空间中，第二个线性层又将输出缩小到原来的1/4，使其与原始输入维度匹配
x = torch.rand(2, 3, 768) 
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


快捷连接（也称为“跳跃连接”或“残差连接”），最初用于计算机视觉中的深度网络（特别是残差网络），目的是缓解梯度消失问题。梯度消失问题指的是在训练过程中，梯度在反向传播时逐渐变小，导致早期网络层难以有效训练。

快捷连接通过跳过一个或多个层，为梯度在网络中的流动提供了一条可替代且更短的路径。这是通过将一层的输出添加到后续层的输出中实现的。

In [10]:
# 一个演示快捷连接的神经网络
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut

        # 5 个层的实现
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            
            # 计算当前层输出
            layer_output = layer(x)
            
            # 检查是否可以使用快捷连接，如果 self.use_shortcut被设置为 True，则会选择性地添加下图的快捷连接
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

# 定义了一个损失函数，用于计算模型输出与用户制定目标的接近程度
def print_gradients(model, x):
    # 前向传播
    output = model(x)
    target = torch.tensor([[0.]])

    # 基于目标和输出之间的差距计算损失
    loss = nn.MSELoss()
    loss = loss(output, target)
    
    # 反向传播计算梯度
    # .backward()方法是 PyTorch 中的一个便捷方法，它可以在模型训练中计算所需的损失梯度
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            
            # 输出其梯度值的平均绝对值
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

<div align="center">
         <img src="../pictures/connect.png" width=70%>
     </div>

In [11]:
# 初始化一个没有快捷连接的神经网络
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

# 指定随机种子用于初始化权重 以确保结果可复现
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173590746708214
layers.1.0.weight has gradient mean of 0.0001201116101583466
layers.2.0.weight has gradient mean of 0.0007152042235247791
layers.3.0.weight has gradient mean of 0.0013988739810883999
layers.4.0.weight has gradient mean of 0.00504964729771018


梯度在从最后一层（layers.4）到第 1 层（layers.0）的过程中逐渐变小，这种现象称为梯度消失问题.

In [12]:
# 有快捷连接
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169791162014008
layers.1.0.weight has gradient mean of 0.20694102346897125
layers.2.0.weight has gradient mean of 0.32896995544433594
layers.3.0.weight has gradient mean of 0.2665732204914093
layers.4.0.weight has gradient mean of 1.3258541822433472


最后一层（layers.4）的梯度仍然大于其他层。然而，梯度值在逐渐接近第 1 层（layers.0） 时趋于稳定，并且没有缩小到几乎消失的程度。

Transformer 块的核心思想是，自注意力机制在多头注意力块中用于识别和分析输入序列中元素之间的关系。相比之下，前馈神经网络则在每个位置上对数据进行单独的修改。这种组合不仅提供了对输入更细致的理解和处理，而且提升了模型处理复杂数据模式的整体能力。

<div align="center">
         <img src="../pictures/Transformer1.png" width=70%>
     </div>

In [13]:
# 定义 GPT Transformer 组件
from previous_chapters import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # 注意力块中添加快捷连接
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # 将原始输入添加回来

        # 在前馈层中添加快捷连接
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 将原始输入添加回来

        return x

这段代码定义了一个 *TransformerBlock* 类，该类在 PyTorch 中实现了一个多头注意力机制（MultiHeadAttention）和一个前馈神经网络（FeedForward），两者都根据提供的配置字典（cfg，比如 GPT_CONFIG_124M）进行配置。
   
层归一化（*LayerNorm*）应用于这两个组件之前，而 *dropout* 应用于这两个组件之后，以便对模型进行正则化并防止过拟合。这种方法也被称为 **前层归一化（Pre-LayerNorm）** 。较早的架构（如最初的 Transformer 模型）在自注意力和前馈神经网络之后才应用层归一化，这种方法被称为 **后层归一化（Post-LayerNorm）**，这通常会导致较差的训练效果。
   
*TransformerBlock* 类还实现了前向传播，其中每个组件后面都跟着一个快捷连接，将块的输入加到其输出上。这个关键特性有助于在训练过程中使梯度在网络中流动，并改善深度模型的学习效果

In [14]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


In [15]:
# 将 DummyTransformerBlock 占位符和 DummyLayerNorm 占位符替换为之前编写的真正的 TransformerBlock 类和 LayerNorm 类
# 从而组装成一个完全可用且参数量为 1.24 亿的 GPT-2 模型

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # 初始化词元和位置嵌入层
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # 创建 TransformerBlock 模块的顺序栈，其层数与 cfg 中指定的层数相同
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # LayerNorm 层
        self.final_norm = LayerNorm(cfg["emb_dim"])

        # 无偏置的线性输出头，将 Transformer 的输出投射到分词器的词汇空间，为词汇中的每个词元生成分数（logits）
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):

        # 接受输入词元索引
        batch_size, seq_len = in_idx.shape

        # 计算嵌入表示
        tok_embeds = self.tok_emb(in_idx)
        
        # 应用位置嵌入
        # device 的设置允许我们在 CPU 或 GPU 上训练模型，具体取决于输入数据所在设备
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)

        # 将序列通过一系列 Transformer 块传递
        x = self.trf_blocks(x)

        # 归一化处理
        x = self.final_norm(x)

        # 计算 logits，这些 logits代表了下一个词元的非归一化概率
        logits = self.out_head(x)
        return logits

In [16]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal number of parameters: {total_params:,}")

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.1381,  0.0077, -0.1963,  ..., -0.0222, -0.1060,  0.1717],
         [ 0.3865, -0.8408, -0.6564,  ..., -0.5163,  0.2369, -0.3357],
         [ 0.6989, -0.1829, -0.1631,  ...,  0.1472, -0.6504, -0.0056],
         [-0.4290,  0.1669, -0.1258,  ...,  1.1579,  0.5303, -0.5549]],

        [[ 0.1094, -0.2894, -0.1467,  ..., -0.0557,  0.2911, -0.2824],
         [ 0.0882, -0.3552, -0.3527,  ...,  1.2930,  0.0053,  0.1898],
         [ 0.6091,  0.4702, -0.4094,  ...,  0.7688,  0.3787, -0.1974],
         [-0.0612, -0.0737,  0.4751,  ...,  1.2463, -0.3834,  0.0609]]],
       grad_fn=<UnsafeViewBackward0>)

Total number of parameters: 163,009,536


要初始化一个参数量为 1.24 亿的 GPT 模型，但是上面代码实际输出的参数量是 1.63 亿的原因：   
原始 GPT-2 架构中使用了一个叫作权重共享（weight tying）的概念。也就是说，原始 GPT-2 架构是将词元嵌入层作为输出层重复使用的，权重共享可以减少模型的总体内存占用和计算复杂度。

In [17]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

# 总参数量 - 输出层
total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])
Number of trainable parameters considering weight tying: 124,412,160


In [18]:
# Exercise 4.1
ff_params = sum(p.numel() for p in block.ff.parameters())
print(f"Total number of parameters in feed forward module: {ff_params:,}")

att_params = sum(p.numel() for p in block.att.parameters())
print(f"Total number of parameters in attention module: {att_params:,}")

Total number of parameters in feed forward module: 4,722,432
Total number of parameters in attention module: 2,360,064


In [19]:
# 计算 GPTModel 对象中 1.63 亿个参数的内存需求（假设每个参数占用 4 字节的 32 位浮点数）
# 计算总的字节大小
total_size_bytes = total_params * 4

# 转换为 MB
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


In [20]:
# Exercise 4.2
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}


def get_config(base_config, model_name="gpt2-small"):
    GPT_CONFIG = base_config.copy()

    if model_name == "gpt2-small":
        GPT_CONFIG["emb_dim"] = 768
        GPT_CONFIG["n_layers"] = 12
        GPT_CONFIG["n_heads"] = 12

    elif model_name == "gpt2-medium":
        GPT_CONFIG["emb_dim"] = 1024
        GPT_CONFIG["n_layers"] = 24
        GPT_CONFIG["n_heads"] = 16

    elif model_name == "gpt2-large":
        GPT_CONFIG["emb_dim"] = 1280
        GPT_CONFIG["n_layers"] = 36
        GPT_CONFIG["n_heads"] = 20

    elif model_name == "gpt2-xl":
        GPT_CONFIG["emb_dim"] = 1600
        GPT_CONFIG["n_layers"] = 48
        GPT_CONFIG["n_heads"] = 25

    else:
        raise ValueError(f"Incorrect model name {model_name}")

    return GPT_CONFIG


def calculate_size(model): # based on chapter code
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total number of parameters: {total_params:,}")

    total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
    print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")
    
    # Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
    total_size_bytes = total_params * 4
    
    # Convert to megabytes
    total_size_mb = total_size_bytes / (1024 * 1024)
    
    print(f"Total size of the model: {total_size_mb:.2f} MB")


for model_abbrev in ("small", "medium", "large", "xl"):
    model_name = f"gpt2-{model_abbrev}"
    CONFIG = get_config(GPT_CONFIG_124M, model_name=model_name)
    model = GPTModel(CONFIG)
    print(f"\n\n{model_name}:")
    calculate_size(model)



gpt2-small:
Total number of parameters: 163,009,536
Number of trainable parameters considering weight tying: 124,412,160
Total size of the model: 621.83 MB


gpt2-medium:
Total number of parameters: 406,212,608
Number of trainable parameters considering weight tying: 354,749,440
Total size of the model: 1549.58 MB


gpt2-large:
Total number of parameters: 838,220,800
Number of trainable parameters considering weight tying: 773,891,840
Total size of the model: 3197.56 MB


gpt2-xl:
Total number of parameters: 1,637,792,000
Number of trainable parameters considering weight tying: 1,557,380,800
Total size of the model: 6247.68 MB


In [21]:
# 生成文本的函数
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx 是当前上下文的索引数组, 其形状为 (batch, n_tokens)
    for _ in range(max_new_tokens):
        
        # 如果当前上下文超过模型支持的长度，则进行截断
        # 例如，如果大语言模型仅支持 5 个词元，但此时文本长度为 10，
        # 则只有最后 5 个词元会被用作输入文本
        idx_cond = idx[:, -context_size:]
        
        # 获取预测结果
        with torch.no_grad():
            logits = model(idx_cond)
        
        # 只关注最后一个时间步的输出内容
        # 形状会从 (batch, n_tokens, vocab_size) 变为 (batch, vocab_size)
        logits = logits[:, -1, :]  

        # 应用 softmax 函数将 logits 转换为概率分布
        # probas 的形状为 (batch, vocab_size)
        probas = torch.softmax(logits, dim=-1)

        # 获取概率值最高的词汇表条目索引
        # idx_next 的形状为 (batch, 1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)

        # 将计算出的下一个索引添加到当前序列中
        # 此时 idx 的形状会变为 (batch, n_tokens + 1)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

语言模型生成循环流程：   
首先循环指定次数以生成新词元，然后将当前上下文裁剪到模型的最大上下文大小，接下来进行预测计算，并根据最高概率选择下一个词元.
   
使用 softmax 函数将 logits 转换为概率分布，并通过 torch.argmax 确定最大值的位置。softmax 函数是单调的，这意味着它在转换为输出时保持了输入的顺序。因此，实际上 softmax 步骤是冗余的，因为 softmax 输出张量中最高分的位置与 logits 张量中的位置是相同的。换句话说，可以直接对 logits 张量应用 torch.argmax 函数，得到相同的结果。不过，我们提供了转换代码来展示将 logits 转换为概率的完整过程，这有助于更好地理解模型如何生成最有可能的下一个词元，这一过程被称为贪心解码.

In [22]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [23]:
model.eval() # 不训练模型时关闭 dropout

out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716, 19470, 39823, 19520, 14689, 20286, 40047]])
Output length: 10


In [24]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

# 乱码是因为还没进行模型训练

Hello, I am lag Jinping Recepushfilled blender


In [25]:
# Exercise 4.3
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_emb": 0.1,        # 嵌入层的 dropout 率
    "drop_rate_attn": 0.1,       # 多注意力头的 dropout 率
    "drop_rate_shortcut": 0.1,   # 快捷连接的 dropout 率
    "qkv_bias": False
}

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate_attn"], # 多注意力头的 dropout 层
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

    def forward(self, x):
        # 注意力块的快捷连接
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut # 将原始输入添加回来

        # 前馈块的快捷连接
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 将原始输入添加回来

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"]) # 嵌入层的 dropout 层

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)